# Fine-tuning on claim + evidence → label, and the English ablation (CheckThat! 2026 Task 2)
### Qwen3-Embedding-0.6B + LoRA · 5 seeds · both T4s · everything inside this notebook

| Part | What it does |
|:--|:--|
| **A. Fine-tuning** (§3) | `Claim: … Evidence 1: …` (full evidence) → Qwen3-Embedding-0.6B with LoRA → True / False / Conflicting. Trained on train, best epoch chosen by validation Macro-F1, 5 seeds. |
| **B. Compact ablation** (§4) | majority vote · vote counts · frozen 8B embeddings with / without instruction · the fine-tuned model · fusion with the votes. Test Macro-F1 with a bootstrap 95% interval. |
| **C. MLP grid** (§5–6) | your 10 configurations (attention / disagreement / verdict expansion), per trace-embedding set, 3 seeds. |

No extra Python file is written and no second process is started: every step runs in a cell, so an error is shown
with its full traceback in the cell where it happened.

**Inputs** (found automatically under `/kaggle/input`; a missing optional input only removes its rows):

| Input | Needed for |
|:--|:--|
| Task data (English `train/validation/test.json`) | everything |
| `{split}_qwen3emb8b_en_v1.npz` (existing claim + trace vectors) | claim-only rows, grid "no instruction" |
| `{split}_claimevid_qwen3emb8b4bit_en_v1.npz` (existing claim + evidence vectors) | claim + evidence, no instruction |
| output of the instruction-embedding notebook | instruction rows, grid "instruction" |
| an earlier version of this notebook (optional) | finished seeds and grid runs are reused |

**Run on Kaggle:** accelerator **GPU T4 ×2**, **internet on**. For a first check set `LIMIT_ROWS = 300` in §1.

## 1. Configuration

In [1]:
from collections import OrderedDict

# ─────────────────────────── PATHS ───────────────────────────
DATA_DIR   = "/kaggle/input/datasets/d202511054/clef-2026-checkthat-task-2-data/English"
INPUT_ROOT = "/kaggle/input"        # searched recursively for embeddings and earlier outputs of this notebook
OUTPUT_DIR = "/kaggle/working"

RUN_FINETUNE = True
RUN_COMPACT  = True
RUN_GRID     = True
LIMIT_ROWS   = None                 # e.g. 300 for a quick trial run; None = everything

# ─────────────────────────── PART A: FINE-TUNING ───────────────────────────
FT_MODEL_ID     = "Qwen/Qwen3-Embedding-0.6B"
FT_TAG          = "ft_ce_qwen3emb06b_lora_v1"     # change it if you change any setting in this block
FT_SEEDS        = [0, 1, 2, 3, 4]
FT_INSTRUCTION  = ("Given a numerical claim and retrieved evidence, classify whether the evidence shows the claim "
                   "to be True, False or Conflicting")       # None = raw text
FT_MAX_SEQ_LENGTH = 8192            # full evidence (longest test row ≈ 5.5k tokens)
FT_EPOCHS       = 4
FT_BATCH_SIZE   = 16                # most rows in one training batch (both GPUs together)
FT_TRAIN_TOKEN_BUDGET = 5120        # most padded tokens PER GPU in one training batch (long rows → fewer rows)
FT_EVAL_TOKEN_BUDGET  = 8192        # most padded tokens PER GPU in one inference batch
FT_GRAD_CHECKPOINTING = True        # keeps training memory small; needed for the long rows
FT_LR_LORA      = 1e-4
FT_LR_HEAD      = 1e-3
FT_WEIGHT_DECAY = 0.01
FT_WARMUP_FRAC  = 0.06
FT_LORA_R, FT_LORA_ALPHA, FT_LORA_DROPOUT = 16, 32, 0.05
FT_LORA_TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
FT_HEAD_DROPOUT = 0.1
FT_CLASS_WEIGHTS = "balanced"       # "balanced" (N / (3·n_c), from train) or None
FT_USE_AMP      = True              # fp16 autocast on the T4s

# ─────────────────────────── TEXT TEMPLATE / DATA FIELDS ───────────────────────────
CLAIM_PREFIX, EVIDENCE_PREFIX, SEPARATOR = "Claim: ", "Evidence {i}: ", "\n\n"
CLAIM_FIELD, EVIDENCE_FIELD, TRACE_FIELD = "claim", "evidences", "Reasoning_traces"
VERDICT_LIST_FIELD, TEST_ID_FIELD, LABEL_KEYS = "Verdict_list", "query_id", ("label", "Label")
CLASS_NAMES = ["False", "True", "Conflicting"]

# ─────────────────────────── PART B: COMPACT ABLATION ───────────────────────────
TRACE_SETS = OrderedDict([("no instruction", "qwen3emb8b_en_v1"), ("instruction", "qwen3emb8b_en_instr_v1")])
CE_SETS = OrderedDict([("no instruction", "claimevid_qwen3emb8b4bit_en_v1"),
                       ("instruction A", "claimevid_qwen3emb8b4bit_en_instrA"),
                       ("instruction B", "claimevid_qwen3emb8b4bit_en_instrB"),
                       ("instruction C", "claimevid_qwen3emb8b4bit_en_instrC")])
PCA_DIM     = 256                   # frozen embeddings → PCA (fit on train) → logistic regression; None = no PCA
LR_C_GRID   = [0.01, 0.1, 1.0]      # chosen by validation Macro-F1
N_BOOTSTRAP = 1000

# ─────────────────────────── PART C: MLP GRID ───────────────────────────
GRID_SEEDS  = [42, 1, 2]
TRACE_MAX, SCALAR_DIM, EXPANDED_SCALAR_DIM = 20, 6, 1024
ATTN_HIDDEN, PROJ_HIDDEN, AE_HIDDEN = 256, 256, 256
HIDDEN_DIMS, DROPOUT, OUTPUT_DIM = [512, 256, 128], 0.3, 3
BATCH_SIZE, MAX_EPOCHS, LEARNING_RATE, WEIGHT_DECAY = 16, 30, 1e-3, 1e-5
GRAD_CLIP_NORM, EARLY_STOP_PATIENCE, LR_PLATEAU_FACTOR, LR_PLATEAU_PATIENCE = 1.0, 6, 0.5, 3
USE_AMP, AE_EPOCHS, AE_LR, AE_BATCH_SIZE = True, 200, 1e-3, 64

## 2. Setup and data

In [2]:
import os, re, gc, sys, json, glob, math, time, copy, random, shutil, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
try:
    from IPython.display import display
except Exception:
    display = print

warnings.filterwarnings("ignore")
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40); pd.set_option("display.max_colwidth", 80)
T_START = time.time()
os.makedirs(OUTPUT_DIR, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
N_GPUS = torch.cuda.device_count()
print(f"PyTorch {torch.__version__} | device {DEVICE} | GPUs {N_GPUS}")

LABEL = {c: i for i, c in enumerate(CLASS_NAMES)}
SPLIT_FILES = OrderedDict(train="train.json", val="validation.json", test="test.json")

def find_one(root, rel):
    if not root or not os.path.isdir(root):
        return None
    hits = [h for h in glob.glob(os.path.join(root, "**", rel), recursive=True)
            if "arab" not in h.lower() and "span" not in h.lower()]
    return min(hits, key=len) if hits else None

def find_input(name):
    """A file by name: OUTPUT_DIR first, then anywhere under INPUT_ROOT."""
    p = os.path.join(OUTPUT_DIR, name)
    return p if os.path.exists(p) else find_one(INPUT_ROOT, name)

def load_json(path):
    with open(path, encoding="utf-8") as f:
        s = f.read()
    try:
        return json.loads(s)
    except json.JSONDecodeError:
        print(f"  {os.path.basename(path)}: invalid JSON, repairing the missing 'Reasoning_traces' key")
        return json.loads(re.sub(r'\{\s*\nx?(\s+)"(?![A-Za-z_]+"\s*:)', r'{\n        "Reasoning_traces": [\n\1"', s))

def get_label(item):
    for k in LABEL_KEYS:
        if k in item:
            return LABEL[item[k]]
    raise KeyError(f"no label key {LABEL_KEYS} in {list(item.keys())}")

def evidence_list(item):
    ev = item.get(EVIDENCE_FIELD) or []
    if isinstance(ev, str):
        ev = [ev]
    return [str(e).strip() for e in ev if str(e).strip()]

def claim_evidence_text(item):
    parts = [CLAIM_PREFIX + item[CLAIM_FIELD].strip()]
    parts += [EVIDENCE_PREFIX.format(i=i) + e for i, e in enumerate(evidence_list(item), 1)]
    text = SEPARATOR.join(parts)
    return text if FT_INSTRUCTION is None else f"Instruct: {FT_INSTRUCTION}\nQuery:{text}"

def trace_texts(item):
    return [t for t in item[TRACE_FIELD] if isinstance(t, str) and t.strip()]

D = OrderedDict()      # split → ids (sorted, as in the embedding files), items, y, vote counts
for split, fname in SPLIT_FILES.items():
    path = find_one(DATA_DIR, fname)
    if path is None:
        raise FileNotFoundError(f"{fname} not found under {DATA_DIR}: attach the data and fix DATA_DIR in §1")
    items = load_json(path)
    ids = [it[TEST_ID_FIELD] for it in items] if split == "test" else list(range(len(items)))
    by_id = dict(zip(ids, items))
    order = sorted(ids)
    if LIMIT_ROWS and LIMIT_ROWS < len(order):          # trial run: a random subset (the files are sorted by label)
        order = sorted(np.random.default_rng(0).choice(len(order), LIMIT_ROWS, replace=False).tolist())
        order = [sorted(ids)[r] for r in order]
    its = [by_id[i] for i in order]
    y = np.array([get_label(it) for it in its], dtype=np.int64)
    cn = np.array([[sum(v == c for v in it[VERDICT_LIST_FIELD]) for c in CLASS_NAMES] for it in its], dtype=np.float64)
    D[split] = {"ids": order, "items": its, "y": y, "counts": cn}
    print(f"{split:5s}: {len(order):5d} rows | labels F/T/C {np.bincount(y, minlength=3).tolist()} | "
          f"trace verdicts F/T/C {(cn.sum(0) / cn.sum()).round(3).tolist()} | per-trace accuracy "
          f"{np.mean(cn[np.arange(len(y)), y] / np.maximum(cn.sum(1), 1)):.3f}")
Y = {s: D[s]["y"] for s in D}
print("\nNote: train / validation traces are much less reliable than test traces, and Conflicting is far more common\n"
      "in train / validation than in test. Validation scores are therefore not on the same scale as test scores.")

PyTorch 2.11.0+cu130 | device cuda | GPUs 2
train:  6400 rows | labels F/T/C [3717, 1175, 1508] | trace verdicts F/T/C [0.399, 0.29, 0.311] | per-trace accuracy 0.480
val  :  1600 rows | labels F/T/C [913, 304, 383] | trace verdicts F/T/C [0.379, 0.301, 0.32] | per-trace accuracy 0.492
test :  2558 rows | labels F/T/C [1783, 520, 255] | trace verdicts F/T/C [0.695, 0.19, 0.115] | per-trace accuracy 0.764

Note: train / validation traces are much less reliable than test traces, and Conflicting is far more common
in train / validation than in test. Validation scores are therefore not on the same scale as test scores.


## 3. Part A: fine-tune Qwen3-Embedding-0.6B on claim + evidence → label
Everything runs inside this notebook, so an error shows its full traceback in the cell that failed.

* **Both GPUs:** the model is wrapped in `nn.DataParallel`, so every batch is split in two halves, one per T4.
* **Memory:** gradient checkpointing is on. It keeps the activations of only one transformer layer at a time, which
  is what makes the long rows (2,300 tokens in train, 5,500 in test) fit in 15 GiB.
* **Batches:** rows of similar length are grouped. A batch holds at most `FT_BATCH_SIZE` rows and at most
  `FT_TRAIN_TOKEN_BUDGET` padded tokens per GPU, so batches of long rows are automatically smaller.
* §3.3 runs the **largest** training batch and the largest inference batch first and prints the peak memory per
  GPU. If memory is too small, it fails there, in the first minute, not hours into training.

### 3.1 Tokenise

In [3]:
import transformers
from transformers import AutoModel, AutoTokenizer
transformers.logging.set_verbosity_error()

tok = AutoTokenizer.from_pretrained(FT_MODEL_ID)
_probe = "Claim: test 123"
_with, _plain = tok(_probe)["input_ids"], tok(_probe, add_special_tokens=False)["input_ids"]
if len(_with) > len(_plain) and _with[:len(_plain)] == _plain:
    EOS = _with[len(_plain):]                         # the tokenizer appends <|endoftext|> itself
else:
    EOS = [tok.get_vocab().get("<|endoftext|>", tok.eos_token_id)]
PAD = tok.pad_token_id if tok.pad_token_id is not None else EOS[-1]
print("end token:", tok.convert_ids_to_tokens(EOS), "| pad id:", PAD)

SEQ = {}                                              # split → list of token-id lists (end token kept)
for s in D:
    texts = [claim_evidence_text(it) for it in D[s]["items"]]
    plain = tok(texts, add_special_tokens=False)["input_ids"]
    keep = FT_MAX_SEQ_LENGTH - len(EOS)
    SEQ[s] = [p[:keep] + EOS for p in plain]
    L = np.array([len(x) for x in SEQ[s]])
    print(f"{s:5s}: {len(L)} rows | tokens p50 {int(np.median(L))} | p90 {int(np.percentile(L, 90))} | max {L.max()} | "
          f"truncated {sum(len(p) > keep for p in plain)}")

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.71k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

end token: ['<|endoftext|>'] | pad id: 151643
train: 6400 rows | tokens p50 198 | p90 334 | max 2324 | truncated 0
val  : 1600 rows | tokens p50 231 | p90 494 | max 4844 | truncated 0
test : 2558 rows | tokens p50 2244 | p90 3416 | max 5553 | truncated 0


### 3.2 Model, batching, prediction

In [4]:
class LoRALinear(nn.Module):
    """A frozen Linear layer plus a trainable low-rank update: W·x + (alpha/r)·B·A·x."""
    def __init__(self, base):
        super().__init__()
        self.base = base
        self.A = nn.Parameter(torch.empty(FT_LORA_R, base.in_features))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, FT_LORA_R))
        self.drop = nn.Dropout(FT_LORA_DROPOUT)
        self.scale = FT_LORA_ALPHA / FT_LORA_R
    def forward(self, x):
        return self.base(x) + F.linear(F.linear(self.drop(x), self.A), self.B) * self.scale

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc = AutoModel.from_pretrained(FT_MODEL_ID, torch_dtype=torch.float32, attn_implementation="sdpa")
        for p in self.enc.parameters():
            p.requires_grad_(False)
        n = 0
        for parent in list(self.enc.modules()):
            for name, child in list(parent.named_children()):
                if isinstance(child, nn.Linear) and name in FT_LORA_TARGETS:
                    setattr(parent, name, LoRALinear(child)); n += 1
        assert n > 0, f"no Linear layer named {FT_LORA_TARGETS} found"
        self.n_lora = n
        if FT_GRAD_CHECKPOINTING:
            self.enc.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
        self.head = nn.Sequential(nn.Dropout(FT_HEAD_DROPOUT), nn.Linear(self.enc.config.hidden_size, 3))
    def forward(self, ids, mask):
        # autocast is entered here (not outside) because DataParallel runs each GPU in its own thread
        with torch.autocast("cuda", dtype=torch.float16, enabled=FT_USE_AMP and ids.is_cuda):
            h = self.enc(input_ids=ids, attention_mask=mask, use_cache=False).last_hidden_state[:, -1]
        h = h.float()                                  # last position = end token (left padding)
        return self.head(h), h

def pad_batch(seqs):
    L = max(len(s) for s in seqs)
    ids = torch.full((len(seqs), L), PAD, dtype=torch.long)
    mask = torch.zeros((len(seqs), L), dtype=torch.long)
    for r, s in enumerate(seqs):                       # left padding
        ids[r, L - len(s):] = torch.tensor(s)
        mask[r, L - len(s):] = 1
    return ids.to(DEVICE), mask.to(DEVICE)

def cut_batches(idx_longest_first, lengths, tokens_per_gpu, max_rows):
    """Consecutive batches with ≤ max_rows rows and ≤ tokens_per_gpu padded tokens on each GPU."""
    n_gpu = max(N_GPUS, 1)
    out, cur = [], []
    for i in idx_longest_first:
        rows_per_gpu = math.ceil((len(cur) + 1) / n_gpu)
        if cur and (len(cur) >= max_rows or rows_per_gpu * lengths[cur[0]] > tokens_per_gpu):
            out.append(cur); cur = []
        cur.append(i)
    if cur:
        out.append(cur)
    return out

def train_batches(lengths, rng):
    """One epoch: shuffle, sort blocks of rows by length, cut into batches, shuffle the batches."""
    perm = rng.permutation(len(lengths))
    batches = []
    for a in range(0, len(perm), FT_BATCH_SIZE * 32):
        block = sorted(perm[a:a + FT_BATCH_SIZE * 32].tolist(), key=lambda i: -lengths[i])
        batches += cut_batches(block, lengths, FT_TRAIN_TOKEN_BUDGET, FT_BATCH_SIZE)
    return [batches[i] for i in rng.permutation(len(batches))]

@torch.inference_mode()
def predict(model, seqs):
    """Probabilities (N, 3) and fine-tuned vectors (N, hidden) for a list of token-id lists."""
    model.eval()
    lengths = [len(s) for s in seqs]
    order = sorted(range(len(seqs)), key=lambda i: -lengths[i])
    probs, embs = np.zeros((len(seqs), 3), np.float32), None
    for b in cut_batches(order, lengths, FT_EVAL_TOKEN_BUDGET, 64):
        ids, mask = pad_batch([seqs[i] for i in b])
        logits, h = model(ids, mask)
        if embs is None:
            embs = np.zeros((len(seqs), h.shape[1]), np.float16)
        probs[b] = torch.softmax(logits.float(), 1).cpu().numpy()
        embs[b] = h.cpu().numpy().astype(np.float16)
    return probs, embs

def gpu_report(title):
    if DEVICE == "cuda":
        print(title, "| peak memory:", " | ".join(f"GPU {g}: {torch.cuda.max_memory_allocated(g) / 2**30:.1f} GiB"
                                                    for g in range(N_GPUS)))
    else:
        print(title)

def f1_per_class(y, p):
    out = []
    for c in range(3):
        tp = np.sum((p == c) & (y == c)); d = 2 * tp + np.sum((p == c) & (y != c)) + np.sum((p != c) & (y == c))
        out.append(2 * tp / d if d else 0.0)
    return np.array(out)

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

_cnt = np.bincount(Y["train"], minlength=3)
FT_W = (torch.tensor(len(Y["train"]) / (3.0 * np.maximum(_cnt, 1)), dtype=torch.float32) if FT_CLASS_WEIGHTS == "balanced"
        else torch.ones(3)).to(DEVICE)
print("class weights F/T/C:", [round(x, 3) for x in FT_W.tolist()])

class weights F/T/C: [0.574, 1.816, 1.415]


### 3.3 Memory check on the largest batches (about a minute)
If this cell fails with an out-of-memory error, lower `FT_TRAIN_TOKEN_BUDGET` (training) or `FT_EVAL_TOKEN_BUDGET`
(inference) in §1. If a single row is too long even alone, lower `FT_MAX_SEQ_LENGTH`.

In [5]:
FT_DIR = os.path.join(OUTPUT_DIR, FT_TAG + ("_trial" if LIMIT_ROWS else ""))
os.makedirs(FT_DIR, exist_ok=True)

def seed_file(seed):
    """Path of a finished seed: in OUTPUT_DIR, or in an attached earlier version of this notebook."""
    name = f"seed{seed}.npz"
    p = os.path.join(FT_DIR, name)
    if os.path.exists(p):
        return p
    hits = glob.glob(os.path.join(INPUT_ROOT, "**", os.path.basename(FT_DIR), name), recursive=True) if INPUT_ROOT else []
    return hits[0] if hits else None

FT_TODO = [s for s in FT_SEEDS if seed_file(s) is None] if RUN_FINETUNE else []
print("seeds to train:", FT_TODO, "| already done:", [s for s in FT_SEEDS if seed_file(s) is not None])

if FT_TODO:
    net = Net().to(DEVICE)
    model = nn.DataParallel(net) if N_GPUS > 1 else net
    print(f"LoRA on {net.n_lora} layers | trainable parameters "
          f"{sum(p.numel() for p in net.parameters() if p.requires_grad):,} | GPUs used: {max(N_GPUS, 1)}")

    lengths = [len(s) for s in SEQ["train"]]
    order = sorted(range(len(lengths)), key=lambda i: -lengths[i])
    b = cut_batches(order, lengths, FT_TRAIN_TOKEN_BUDGET, FT_BATCH_SIZE)[0]          # the largest training batch
    model.train()
    ids, mask = pad_batch([SEQ["train"][i] for i in b])
    t = time.time()
    logits, _ = model(ids, mask)
    loss = F.cross_entropy(logits.float(), torch.tensor(Y["train"][b], dtype=torch.long, device=DEVICE), weight=FT_W)
    loss.backward()
    gpu_report(f"training : {len(b)} rows × {ids.shape[1]} tokens, {time.time() - t:.1f}s")
    net.zero_grad(set_to_none=True)
    del ids, mask, logits, loss
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
        for g in range(N_GPUS): torch.cuda.reset_peak_memory_stats(g)

    longest = max(D, key=lambda s: max(len(x) for x in SEQ[s]))
    lengths = [len(x) for x in SEQ[longest]]
    order = sorted(range(len(lengths)), key=lambda i: -lengths[i])
    b = cut_batches(order, lengths, FT_EVAL_TOKEN_BUDGET, 64)[0]                      # the largest inference batch
    t = time.time()
    predict(model, [SEQ[longest][i] for i in b])
    gpu_report(f"inference: {len(b)} rows × {lengths[b[0]]} tokens ({longest}), {time.time() - t:.1f}s")
    del net, model
    gc.collect()
    if DEVICE == "cuda": torch.cuda.empty_cache()

seeds to train: [0, 1, 2, 3, 4] | already done: []


model.safetensors: reconstructing file:   0%|          |  0.00B / 1.19GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

LoRA on 196 layers | trainable parameters 10,095,619 | GPUs used: 2
training : 4 rows × 2324 tokens, 5.5s | peak memory: GPU 0: 6.8 GiB | GPU 1: 3.0 GiB
inference: 2 rows × 5553 tokens (test), 4.9s | peak memory: GPU 0: 7.0 GiB | GPU 1: 2.7 GiB


### 3.4 Train
One seed after the other. After each epoch the validation Macro-F1 is computed, and the weights of the best epoch
are used for the final predictions. Each finished seed is saved at once (`<FT_TAG>/seed<k>.npz` and `.json`).

In [6]:
for seed in FT_TODO:
    print(f"\n================ seed {seed} ================")
    set_seed(seed)
    net = Net().to(DEVICE)
    model = nn.DataParallel(net) if N_GPUS > 1 else net
    lora_params = [p for n, p in net.named_parameters() if p.requires_grad and not n.startswith("head")]
    opt = torch.optim.AdamW([{"params": lora_params, "lr": FT_LR_LORA},
                             {"params": net.head.parameters(), "lr": FT_LR_HEAD}], weight_decay=FT_WEIGHT_DECAY)
    scaler = torch.amp.GradScaler("cuda", enabled=FT_USE_AMP and DEVICE == "cuda")
    seqs, ytr = SEQ["train"], Y["train"]
    lengths = [len(s) for s in seqs]
    rng = np.random.default_rng(seed)
    total_steps = len(train_batches(lengths, np.random.default_rng(seed))) * FT_EPOCHS
    warm = max(1, int(FT_WARMUP_FRAC * total_steps))
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: (s + 1) / warm if s < warm else max(0.0, (total_steps - s) / max(1, total_steps - warm)))

    best_f1, best_epoch, best_state, history, t0 = -1.0, 0, None, [], time.time()
    for ep in range(1, FT_EPOCHS + 1):
        model.train()
        batches = train_batches(lengths, rng)
        run_loss = 0.0
        for step, b in enumerate(batches, 1):
            ids, mask = pad_batch([seqs[i] for i in b])
            y = torch.tensor(ytr[b], dtype=torch.long, device=DEVICE)
            logits, _ = model(ids, mask)
            loss = F.cross_entropy(logits.float(), y, weight=FT_W)
            if not torch.isfinite(loss):
                raise RuntimeError(f"non-finite loss at epoch {ep}, step {step} "
                                   f"({len(b)} rows × {ids.shape[1]} tokens). Set FT_USE_AMP = False in §1.")
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_([p for p in net.parameters() if p.requires_grad], 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            run_loss += loss.item()
            if step % 50 == 0 or step == len(batches):
                print(f"  epoch {ep} step {step:4d}/{len(batches)} | loss {run_loss / step:.4f} | "
                      f"{(time.time() - t0) / 60:.1f} min", flush=True)
        pv, _ = predict(model, SEQ["val"])
        f1 = float(f1_per_class(Y["val"], pv.argmax(1)).mean())
        history.append({"epoch": ep, "train_loss": run_loss / len(batches), "val_macro_f1": f1})
        if f1 > best_f1:
            best_f1, best_epoch = f1, ep
            best_state = {k: v.detach().cpu().clone() for k, v in net.named_parameters() if v.requires_grad}
        print(f"epoch {ep}: train loss {run_loss / len(batches):.4f} | val Macro-F1 {f1:.4f}"
              f"{'  ← best' if best_epoch == ep else ''}", flush=True)

    net.load_state_dict(best_state, strict=False)                 # back to the best epoch
    arrays, metrics = {}, {}
    for s in D:
        p, e = predict(model, SEQ[s])
        arrays[f"probs_{s}"], arrays[f"emb_{s}"] = p, e
        arrays[f"ids_{s}"] = np.array([str(i) for i in D[s]["ids"]])
        f = f1_per_class(Y[s], p.argmax(1))
        metrics[s] = {"macro_f1": float(f.mean()), "acc": float(np.mean(p.argmax(1) == Y[s])), "f1_per_class": f.tolist()}
        print(f"[{s:5s}] Macro-F1 {f.mean():.4f} | acc {metrics[s]['acc']:.4f} | F1 F/T/C {f.round(3).tolist()}")
    np.savez(os.path.join(FT_DIR, f"seed{seed}.npz"), **arrays)
    with open(os.path.join(FT_DIR, f"seed{seed}.json"), "w") as fjson:
        json.dump({"seed": seed, "best_epoch": best_epoch, "history": history, "metrics": metrics,
                   "minutes": (time.time() - t0) / 60}, fjson, indent=2)
    gpu_report(f"seed {seed} done in {(time.time() - t0) / 60:.1f} min")
    del net, model, opt, best_state
    gc.collect()
    if DEVICE == "cuda": torch.cuda.empty_cache()


================ seed 0 ================


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

  epoch 1 step   50/414 | loss 1.5524 | 1.3 min
  epoch 1 step  100/414 | loss 1.3678 | 2.5 min
  epoch 1 step  150/414 | loss 1.3191 | 3.9 min
  epoch 1 step  200/414 | loss 1.2541 | 5.1 min
  epoch 1 step  250/414 | loss 1.2031 | 6.4 min
  epoch 1 step  300/414 | loss 1.1671 | 7.7 min
  epoch 1 step  350/414 | loss 1.1408 | 8.9 min
  epoch 1 step  400/414 | loss 1.1165 | 10.1 min
  epoch 1 step  414/414 | loss 1.1137 | 10.5 min
epoch 1: train loss 1.1137 | val Macro-F1 0.4412  ← best
  epoch 2 step   50/414 | loss 0.8587 | 12.4 min
  epoch 2 step  100/414 | loss 0.8559 | 13.7 min
  epoch 2 step  150/414 | loss 0.8568 | 15.0 min
  epoch 2 step  200/414 | loss 0.8615 | 16.3 min
  epoch 2 step  250/414 | loss 0.8610 | 17.6 min
  epoch 2 step  300/414 | loss 0.8586 | 18.8 min
  epoch 2 step  350/414 | loss 0.8514 | 20.0 min
  epoch 2 step  400/414 | loss 0.8464 | 21.3 min
  epoch 2 step  414/414 | loss 0.8450 | 21.7 min
epoch 2: train loss 0.8450 | val Macro-F1 0.5264  ← best
  epoch 3 s

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

  epoch 1 step   50/413 | loss 1.4077 | 1.3 min
  epoch 1 step  100/413 | loss 1.2704 | 2.6 min
  epoch 1 step  150/413 | loss 1.2329 | 3.8 min
  epoch 1 step  200/413 | loss 1.2034 | 5.1 min
  epoch 1 step  250/413 | loss 1.1616 | 6.4 min
  epoch 1 step  300/413 | loss 1.1426 | 7.7 min
  epoch 1 step  350/413 | loss 1.1245 | 8.9 min
  epoch 1 step  400/413 | loss 1.1047 | 10.1 min
  epoch 1 step  413/413 | loss 1.1010 | 10.5 min
epoch 1: train loss 1.1010 | val Macro-F1 0.5010  ← best
  epoch 2 step   50/415 | loss 0.9127 | 12.5 min
  epoch 2 step  100/415 | loss 0.9122 | 13.7 min
  epoch 2 step  150/415 | loss 0.8767 | 15.0 min
  epoch 2 step  200/415 | loss 0.8800 | 16.1 min
  epoch 2 step  250/415 | loss 0.8693 | 17.4 min
  epoch 2 step  300/415 | loss 0.8637 | 18.7 min
  epoch 2 step  350/415 | loss 0.8610 | 20.0 min
  epoch 2 step  400/415 | loss 0.8462 | 21.2 min
  epoch 2 step  415/415 | loss 0.8447 | 21.7 min
epoch 2: train loss 0.8447 | val Macro-F1 0.5259  ← best
  epoch 3 s

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

  epoch 1 step   50/414 | loss 1.3827 | 1.3 min
  epoch 1 step  100/414 | loss 1.3790 | 2.6 min
  epoch 1 step  150/414 | loss 1.3029 | 3.9 min
  epoch 1 step  200/414 | loss 1.2638 | 5.2 min
  epoch 1 step  250/414 | loss 1.2203 | 6.4 min
  epoch 1 step  300/414 | loss 1.1848 | 7.7 min
  epoch 1 step  350/414 | loss 1.1461 | 8.9 min
  epoch 1 step  400/414 | loss 1.1232 | 10.2 min
  epoch 1 step  414/414 | loss 1.1235 | 10.5 min
epoch 1: train loss 1.1235 | val Macro-F1 0.5564  ← best
  epoch 2 step   50/414 | loss 0.8492 | 12.5 min
  epoch 2 step  100/414 | loss 0.8579 | 13.7 min
  epoch 2 step  150/414 | loss 0.8533 | 15.0 min
  epoch 2 step  200/414 | loss 0.8581 | 16.3 min
  epoch 2 step  250/414 | loss 0.8504 | 17.5 min
  epoch 2 step  300/414 | loss 0.8462 | 18.9 min
  epoch 2 step  350/414 | loss 0.8370 | 20.1 min
  epoch 2 step  400/414 | loss 0.8447 | 21.4 min
  epoch 2 step  414/414 | loss 0.8478 | 21.7 min
epoch 2: train loss 0.8478 | val Macro-F1 0.5393
  epoch 3 step   50

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

  epoch 1 step   50/415 | loss 1.4531 | 1.3 min
  epoch 1 step  100/415 | loss 1.3215 | 2.6 min
  epoch 1 step  150/415 | loss 1.2920 | 4.0 min
  epoch 1 step  200/415 | loss 1.2427 | 5.2 min
  epoch 1 step  250/415 | loss 1.1960 | 6.5 min
  epoch 1 step  300/415 | loss 1.1624 | 7.8 min
  epoch 1 step  350/415 | loss 1.1368 | 9.0 min
  epoch 1 step  400/415 | loss 1.1191 | 10.2 min
  epoch 1 step  415/415 | loss 1.1151 | 10.6 min
epoch 1: train loss 1.1151 | val Macro-F1 0.4374  ← best
  epoch 2 step   50/415 | loss 0.8325 | 12.6 min
  epoch 2 step  100/415 | loss 0.8336 | 13.9 min
  epoch 2 step  150/415 | loss 0.8355 | 15.2 min
  epoch 2 step  200/415 | loss 0.8337 | 16.5 min
  epoch 2 step  250/415 | loss 0.8303 | 17.7 min
  epoch 2 step  300/415 | loss 0.8345 | 19.0 min
  epoch 2 step  350/415 | loss 0.8283 | 20.3 min
  epoch 2 step  400/415 | loss 0.8290 | 21.5 min
  epoch 2 step  415/415 | loss 0.8276 | 21.9 min
epoch 2: train loss 0.8276 | val Macro-F1 0.5841  ← best
  epoch 3 s

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

  epoch 1 step   50/414 | loss 1.4048 | 1.3 min
  epoch 1 step  100/414 | loss 1.3287 | 2.6 min
  epoch 1 step  150/414 | loss 1.2557 | 3.8 min
  epoch 1 step  200/414 | loss 1.2075 | 5.0 min
  epoch 1 step  250/414 | loss 1.1737 | 6.3 min
  epoch 1 step  300/414 | loss 1.1390 | 7.6 min
  epoch 1 step  350/414 | loss 1.1153 | 8.8 min
  epoch 1 step  400/414 | loss 1.0972 | 10.1 min
  epoch 1 step  414/414 | loss 1.0898 | 10.4 min
epoch 1: train loss 1.0898 | val Macro-F1 0.5404  ← best
  epoch 2 step   50/415 | loss 0.8490 | 12.4 min
  epoch 2 step  100/415 | loss 0.8765 | 13.6 min
  epoch 2 step  150/415 | loss 0.8431 | 14.8 min
  epoch 2 step  200/415 | loss 0.8391 | 16.1 min
  epoch 2 step  250/415 | loss 0.8410 | 17.4 min
  epoch 2 step  300/415 | loss 0.8342 | 18.7 min
  epoch 2 step  350/415 | loss 0.8323 | 20.0 min
  epoch 2 step  400/415 | loss 0.8242 | 21.3 min
  epoch 2 step  415/415 | loss 0.8202 | 21.6 min
epoch 2: train loss 0.8202 | val Macro-F1 0.5808  ← best
  epoch 3 s

In [7]:
# Load every finished seed (from this run or from an attached earlier version)
FT = {}
for s in FT_SEEDS:
    p = seed_file(s)
    if p is None:
        continue
    with np.load(p) as z:
        for sp in D:
            assert list(z[f"ids_{sp}"]) == [str(i) for i in D[sp]["ids"]], f"seed {s}: row ids differ from the data"
        FT[s] = {sp: z[f"probs_{sp}"].astype(np.float64) for sp in D}
        FT[s]["emb"] = {sp: z[f"emb_{sp}"].astype(np.float32) for sp in D} if s == FT_SEEDS[0] else None
    if os.path.abspath(os.path.dirname(p)) != os.path.abspath(FT_DIR):        # keep it in this version's output too
        shutil.copy(p, FT_DIR)
if FT:
    display(pd.DataFrame([{"seed": s,
                           "val Macro-F1": f1_per_class(Y["val"], r["val"].argmax(1)).mean(),
                           "test Macro-F1": f1_per_class(Y["test"], r["test"].argmax(1)).mean(),
                           "test acc": np.mean(r["test"].argmax(1) == Y["test"]),
                           "test F1 Conflicting": f1_per_class(Y["test"], r["test"].argmax(1))[2]}
                          for s, r in FT.items()]).round(4))
else:
    print("no fine-tuned seeds available: the fine-tuned rows are left out of the ablation")

,seed,val Macro-F1,test Macro-F1,test acc,test F1 Conflicting
0,0,0.5785,0.4873,0.7267,0.2161
1,1,0.5925,0.4475,0.7205,0.1747
2,2,0.5774,0.5040,0.7510,0.1353
3,3,0.5841,0.4382,0.7267,0.1363
4,4,0.5808,0.3885,0.7095,0.1384


## 4. Part B: compact ablation
* **Frozen embeddings:** PCA (fit on train) → standardise → class-balanced logistic regression fitted on train;
  `C` chosen by validation Macro-F1.
* **Fusion:** a class-balanced logistic regression on `[vote shares, log-probabilities of the component models]`,
  fitted on **validation** (the components were fitted on train, so their validation probabilities are honest).
  Its validation score is 5-fold cross-validated; the "votes only" fusion row is the control for this protocol.
* **Test interval:** bootstrap over test rows (95%). For the multi-seed row, the ± is the standard deviation over seeds.

In [8]:
def f1s(y, p):
    out = []
    for c in range(3):
        tp = np.sum((p == c) & (y == c)); d = 2 * tp + np.sum((p == c) & (y != c)) + np.sum((p != c) & (y == c))
        out.append(2 * tp / d if d else 0.0)
    return np.array(out)
def mf1(y, p): return float(f1s(y, p).mean())
def boot_ci(y, p, n=N_BOOTSTRAP):
    rng = np.random.default_rng(0)
    v = [mf1(y[i], p[i]) for i in rng.integers(0, len(y), (n, len(y)))]
    return float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))

def majority(cn):
    """Plurality of the trace verdicts; ties → Conflicting (as in the earlier notebooks)."""
    p = cn.argmax(1)
    p[(cn == cn.max(1, keepdims=True)).sum(1) > 1] = 2
    return p

def vote_feats(cn):
    n = np.maximum(cn.sum(1, keepdims=True), 1); sh = cn / n
    ent = -(sh * np.log(sh + 1e-9)).sum(1, keepdims=True); srt = np.sort(sh, 1)
    return np.hstack([sh, np.log1p(cn) / np.log(21), (cn == 0).astype(float), ent, (srt[:, 2] - srt[:, 1])[:, None]])

def fit_lr(Xtr, ytr, C=1.0):
    return make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=2000, class_weight="balanced")).fit(Xtr, ytr)

def emb_lr(E):
    """E: split → (N, d). Returns val / test probabilities of the LR with the best validation Macro-F1."""
    X = E
    if PCA_DIM and E["train"].shape[1] > PCA_DIM and len(E["train"]) > PCA_DIM:
        pca = PCA(PCA_DIM, random_state=0).fit(E["train"])
        X = {s: pca.transform(E[s]) for s in E}
    best = None
    for C in LR_C_GRID:
        m = fit_lr(X["train"], Y["train"], C)
        f = mf1(Y["val"], m.predict(X["val"]))
        if best is None or f > best[0]:
            best = (f, C, m)
    return {"val": best[2].predict_proba(X["val"]), "test": best[2].predict_proba(X["test"]), "note": f"C={best[1]}"}

def load_ce(tag):
    out = {}
    for s in D:
        p = find_input(f"{s}_{tag}.npz")
        if p is None:
            return None
        with np.load(p, allow_pickle=False) as z:
            row = {i: r for r, i in enumerate(z["__ids__"].tolist())}
            out[s] = z["emb"][[row[str(i)] for i in D[s]["ids"]]].astype(np.float32)
    return out

def load_claim_rows(tag):
    out = {}
    for s in D:
        p = find_input(f"{s}_{tag}.npz")
        if p is None:
            return None
        with np.load(p, allow_pickle=False) as z:
            out[s] = np.stack([z[f"id_{i}"][0] for i in D[s]["ids"]]).astype(np.float32)
    return out

ROWS, PROBS = [], OrderedDict()      # PROBS: name → {"val": (N,3), "test": (N,3)} for fusion

def add_row(group, name, val_pred, test_pred, note="", val_f1=None):
    yt = Y["test"]; lo, hi = boot_ci(yt, test_pred); f = f1s(yt, test_pred)
    ROWS.append({"Group": group, "Method": name,
                 "Val Macro-F1": mf1(Y["val"], val_pred) if val_f1 is None else val_f1,
                 "Test Macro-F1": float(f.mean()), "Test 95% CI / ±std": f"[{lo:.3f}, {hi:.3f}]",
                 "Test Acc": float(np.mean(test_pred == yt)), "F1 False": f[0], "F1 True": f[1], "F1 Conflicting": f[2],
                 "Note": note})
    print(f"  {group:10s} | {name:48s} | val {ROWS[-1]['Val Macro-F1']:.4f} | test {f.mean():.4f}", flush=True)

if RUN_COMPACT:
    CN = {s: D[s]["counts"] for s in D}
    # ---- votes ----
    add_row("Votes", "Majority vote of the traces", majority(CN["val"]), majority(CN["test"]), "reference")
    m = fit_lr(vote_feats(CN["train"]), Y["train"])
    PROBS["votes"] = {s: m.predict_proba(vote_feats(CN[s])) for s in ("val", "test")}
    add_row("Votes", "Vote counts → LR", PROBS["votes"]["val"].argmax(1), PROBS["votes"]["test"].argmax(1))
    # ---- frozen embeddings ----
    for label, tag in TRACE_SETS.items():
        E = load_claim_rows(tag)
        if E is None:
            print(f"  (claim-only, {label}: {tag} not found, row skipped)"); continue
        r = emb_lr(E); PROBS[f"claim / {label}"] = r
        add_row("Frozen 8B", f"Claim only, {label} → LR", r["val"].argmax(1), r["test"].argmax(1), r["note"]); del E
    CE_VAL = {}
    for label, tag in CE_SETS.items():
        E = load_ce(tag)
        if E is None:
            print(f"  (claim + evidence, {label}: {tag} not found, row skipped)"); continue
        r = emb_lr(E); PROBS[f"C+E / {label}"] = r; CE_VAL[label] = mf1(Y["val"], r["val"].argmax(1))
        add_row("Frozen 8B", f"Claim + evidence, {label} → LR", r["val"].argmax(1), r["test"].argmax(1), r["note"]); del E
    # ---- fine-tuned ----
    if FT:
        seeds = list(FT)
        tf = np.array([mf1(Y["test"], FT[s]["test"].argmax(1)) for s in seeds])
        vf = np.array([mf1(Y["val"], FT[s]["val"].argmax(1)) for s in seeds])
        pc = np.array([f1s(Y["test"], FT[s]["test"].argmax(1)) for s in seeds]).mean(0)
        ROWS.append({"Group": "Fine-tuned", "Method": f"Claim + evidence, 0.6B LoRA (mean of {len(seeds)} seeds)",
                     "Val Macro-F1": vf.mean(), "Test Macro-F1": tf.mean(), "Test 95% CI / ±std": f"± {tf.std():.3f}",
                     "Test Acc": float(np.mean([np.mean(FT[s]["test"].argmax(1) == Y["test"]) for s in seeds])),
                     "F1 False": pc[0], "F1 True": pc[1], "F1 Conflicting": pc[2],
                     "Note": f"min {tf.min():.3f}, max {tf.max():.3f}"})
        PROBS["C+E / fine-tuned"] = {s: np.mean([FT[k][s] for k in seeds], 0) for s in ("val", "test")}
        add_row("Fine-tuned", f"Claim + evidence, 0.6B LoRA (ensemble of {len(seeds)} seeds)",
                PROBS["C+E / fine-tuned"]["val"].argmax(1), PROBS["C+E / fine-tuned"]["test"].argmax(1))
        if FT[seeds[0]]["emb"] is not None:
            r = emb_lr({"train": FT[seeds[0]]["emb"]["train"], "val": FT[seeds[0]]["emb"]["val"], "test": FT[seeds[0]]["emb"]["test"]})
            add_row("Fine-tuned", f"Fine-tuned vectors (seed {seeds[0]}) → LR", r["val"].argmax(1), r["test"].argmax(1), r["note"])
    # ---- fusion (stacker fitted on validation) ----
    def fuse(name, comps):
        def X(s): return np.hstack([CN[s] / np.maximum(CN[s].sum(1, keepdims=True), 1)] +
                                   [np.log(PROBS[c][s] + 1e-6) for c in comps])
        st = make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=2000, class_weight="balanced"))
        k = int(min(5, np.bincount(Y["val"], minlength=3).min()))
        cv = cross_val_predict(st, X("val"), Y["val"], cv=StratifiedKFold(k, shuffle=True, random_state=0)) if k >= 2 \
            else st.fit(X("val"), Y["val"]).predict(X("val"))
        add_row("Fusion", name, cv, st.fit(X("val"), Y["val"]).predict(X("test")), "fitted on val; val score is 5-fold CV")
    fuse("Votes only (control)", [])
    best_instr = max((k for k in CE_VAL if k != "no instruction"), key=lambda k: CE_VAL[k], default=None)
    if "C+E / no instruction" in PROBS: fuse("Votes + C+E no instruction", ["C+E / no instruction"])
    if best_instr: fuse(f"Votes + C+E {best_instr} (best on val)", [f"C+E / {best_instr}"])
    if "C+E / fine-tuned" in PROBS:
        fuse("Votes + C+E fine-tuned", ["C+E / fine-tuned"])
        frozen = f"C+E / {best_instr}" if best_instr else ("C+E / no instruction" if "C+E / no instruction" in PROBS else None)
        if frozen: fuse(f"Votes + fine-tuned + {frozen}", ["C+E / fine-tuned", frozen])

    COMPACT = pd.DataFrame(ROWS)
    _ref = COMPACT.loc[0, "Test Macro-F1"]
    COMPACT.insert(4, "Δ vs majority", COMPACT["Test Macro-F1"] - _ref)
    _num = [c for c in COMPACT.columns if COMPACT[c].dtype.kind == "f"]
    try:
        display(COMPACT.style.format({c: "{:.4f}" for c in _num} | {"Δ vs majority": "{:+.4f}"})
                .highlight_max(subset=["Test Macro-F1", "F1 Conflicting"], color="#c8e6c9"))
    except Exception:
        print(COMPACT.round(4).to_string())
    COMPACT.to_csv(os.path.join(OUTPUT_DIR, "compact_ablation.csv"), index=False)
    np.savez(os.path.join(OUTPUT_DIR, "compact_ablation_probs.npz"),
             **{f"{k}|{s}": v[s] for k, v in PROBS.items() for s in ("val", "test")},
             y_val=Y["val"], y_test=Y["test"])
    print("\nRead it like this: a row helps only if its test interval lies above the majority-vote row, "
          "and F1 Conflicting shows whether the gain is on the hard class.")

  Votes      | Majority vote of the traces                      | val 0.4690 | test 0.5967
  Votes      | Vote counts → LR                                 | val 0.3720 | test 0.5961
  Frozen 8B  | Claim only, no instruction → LR                  | val 0.5507 | test 0.4684
  (claim-only, instruction: qwen3emb8b_en_instr_v1 not found, row skipped)
  Frozen 8B  | Claim + evidence, no instruction → LR            | val 0.5448 | test 0.4192
  (claim + evidence, instruction A: claimevid_qwen3emb8b4bit_en_instrA not found, row skipped)
  (claim + evidence, instruction B: claimevid_qwen3emb8b4bit_en_instrB not found, row skipped)
  (claim + evidence, instruction C: claimevid_qwen3emb8b4bit_en_instrC not found, row skipped)
  Fine-tuned | Claim + evidence, 0.6B LoRA (ensemble of 5 seeds) | val 0.6089 | test 0.4689
  Fine-tuned | Fine-tuned vectors (seed 0) → LR                 | val 0.5599 | test 0.4492
  Fusion     | Votes only (control)                             | val 0.4672 | test 0.5974
  

,Group,Method,Val Macro-F1,Test Macro-F1,Δ vs majority,Test 95% CI / ±std,Test Acc,F1 False,F1 True,F1 Conflicting,Note
0,Votes,Majority vote of the traces,0.4690,0.5967,+0.0000,"[0.575, 0.617]",0.7721,0.8769,0.6979,0.2152,reference
1,Votes,Vote counts → LR,0.3720,0.5961,-0.0006,"[0.575, 0.616]",0.7776,0.8777,0.7094,0.2012,
2,Frozen 8B,"Claim only, no instruction → LR",0.5507,0.4684,-0.1283,"[0.444, 0.489]",0.6247,0.7671,0.4165,0.2215,C=0.1
3,Frozen 8B,"Claim + evidence, no instruction → LR",0.5448,0.4192,-0.1775,"[0.395, 0.441]",0.6611,0.8212,0.2570,0.1793,C=0.01
4,Fine-tuned,"Claim + evidence, 0.6B LoRA (mean of 5 seeds)",0.5827,0.4531,-0.1436,± 0.040,0.7269,0.8508,0.3483,0.1601,"min 0.388, max 0.504"
5,Fine-tuned,"Claim + evidence, 0.6B LoRA (ensemble of 5 seeds)",0.6089,0.4689,-0.1277,"[0.443, 0.492]",0.7385,0.8569,0.3772,0.1727,
6,Fine-tuned,Fine-tuned vectors (seed 0) → LR,0.5599,0.4492,-0.1474,"[0.426, 0.471]",0.7236,0.8404,0.3739,0.1333,C=0.01
7,Fusion,Votes only (control),0.4672,0.5974,+0.0008,"[0.576, 0.618]",0.7783,0.8781,0.7058,0.2084,fitted on val; val score is 5-fold CV
8,Fusion,Votes + C+E no instruction,0.5607,0.5023,-0.0943,"[0.480, 0.525]",0.7377,0.8640,0.4649,0.1781,fitted on val; val score is 5-fold CV
9,Fusion,Votes + C+E fine-tuned,0.5968,0.5930,-0.0037,"[0.568, 0.614]",0.7952,0.8925,0.6772,0.2094,fitted on val; val score is 5-fold CV



Read it like this: a row helps only if its test interval lies above the majority-vote row, and F1 Conflicting shows whether the gain is on the hard class.


## 5. Part C: 10-config MLP grid, per trace-embedding set
Same model as `mlp-english-2` (`[claim ; attention-pooled traces ; trace std ; verdict features] → MLP`), with the
two fixes listed at the top, run with `GRID_SEEDS` seeds for each available trace-embedding set.
Verdict features (6): vote shares F / T / C, majority share, distinct-verdict ratio, entropy.

In [9]:
def verdict_features(cn):
    tot = cn.sum()
    if tot == 0:
        return np.zeros(SCALAR_DIM, np.float32)
    sh = cn / tot
    ent = -sum(p * np.log(p + 1e-9) for p in sh if p > 0)
    return np.array([sh[0], sh[1], sh[2], sh.max(), (cn > 0).sum() / 3.0, ent], np.float32)

def build_arrays(tag):
    """split → tensors (claim, traces, mask, scalars, labels) from {split}_<tag>.npz; None if a file is missing."""
    paths = {s: find_input(f"{s}_{tag}.npz") for s in D}
    if any(p is None for p in paths.values()):
        return None
    ARR = {}
    for s in D:
        N = len(D[s]["ids"])
        with np.load(paths[s], allow_pickle=False) as z:
            dim = z[f"id_{D[s]['ids'][0]}"].shape[1]
            claims = np.zeros((N, dim), np.float32); traces = np.zeros((N, TRACE_MAX, dim), np.float32)
            masks = np.zeros((N, TRACE_MAX), np.float32); bad = 0
            for r, (i, it) in enumerate(zip(D[s]["ids"], D[s]["items"])):
                a = z[f"id_{i}"]
                bad += (a.shape[0] - 1) != len(trace_texts(it))
                n = min(a.shape[0] - 1, TRACE_MAX)
                claims[r] = a[0]; traces[r, :n] = a[1:1 + n]; masks[r, :n] = 1
        if bad:
            raise RuntimeError(f"{tag} / {s}: {bad} rows have a trace count different from the JSON")
        scal = np.stack([verdict_features(c) for c in D[s]["counts"]])
        ARR[s] = {"claim": torch.from_numpy(claims), "traces": torch.from_numpy(traces), "mask": torch.from_numpy(masks),
                  "scalars": torch.from_numpy(scal), "labels": torch.from_numpy(D[s]["y"].astype(np.int64))}
        print(f"  {tag} / {s}: {N} rows, dim {dim}, traces used mean {masks.sum(1).mean():.2f}", flush=True)
    return ARR

class AttentionPool(nn.Module):
    def __init__(self, dim, hidden=ATTN_HIDDEN):
        super().__init__()
        self.attn = nn.Sequential(nn.Linear(dim, hidden), nn.Tanh(), nn.Linear(hidden, 1))
    def forward(self, traces, mask):
        scores  = self.attn(traces).squeeze(-1).masked_fill(mask == 0, float("-inf"))
        has_any = mask.sum(dim=1, keepdim=True) > 0
        scores  = torch.where(has_any, scores, torch.zeros_like(scores))
        weights = torch.softmax(scores, dim=1) * has_any
        return torch.bmm(weights.unsqueeze(1), traces).squeeze(1)

def masked_std(traces, mask, eps=1e-6):
    mask_f = mask.unsqueeze(-1)
    count  = mask_f.sum(dim=1).clamp(min=1.0)
    mean   = (traces * mask_f).sum(dim=1) / count
    var    = ((traces - mean.unsqueeze(1)) ** 2 * mask_f).sum(dim=1) / count
    return torch.sqrt(var + eps)

class CyclicDuplication(nn.Module):
    def __init__(self, in_dim=SCALAR_DIM, out_dim=EXPANDED_SCALAR_DIM):
        super().__init__()
        self.out_dim, self.repeats = out_dim, math.ceil(out_dim / in_dim)
    def forward(self, x):
        return x.repeat(1, self.repeats)[:, :self.out_dim]

class LearnableProjection(nn.Module):
    def __init__(self, in_dim=SCALAR_DIM, hidden=PROJ_HIDDEN, out_dim=EXPANDED_SCALAR_DIM):
        super().__init__()
        self.proj = nn.Sequential(nn.Linear(in_dim, hidden), nn.BatchNorm1d(hidden), nn.ReLU(),
                                  nn.Linear(hidden, out_dim), nn.BatchNorm1d(out_dim), nn.ReLU())
    def forward(self, x): return self.proj(x)

class VerdictAutoencoder(nn.Module):
    def __init__(self, in_dim=SCALAR_DIM, hidden=AE_HIDDEN, latent=EXPANDED_SCALAR_DIM):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(in_dim, hidden), nn.ReLU(), nn.Linear(hidden, latent), nn.ReLU())
        self.decoder = nn.Sequential(nn.Linear(latent, hidden), nn.ReLU(), nn.Linear(hidden, in_dim))
    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z), z

class FrozenAEEncoder(nn.Module):
    def __init__(self, ae):
        super().__init__()
        self.encoder = copy.deepcopy(ae.encoder)
        for p in self.encoder.parameters(): p.requires_grad_(False)
    def train(self, mode=True): return super().train(False)
    def forward(self, x): return self.encoder(x)

VERDICT_DIMS = {"raw": SCALAR_DIM, "dup": EXPANDED_SCALAR_DIM, "ae": EXPANDED_SCALAR_DIM, "proj": EXPANDED_SCALAR_DIM}
VERDICT_NAMES = {"raw": f"Raw {SCALAR_DIM}", "dup": "Dup_1024", "ae": "AE_1024", "proj": "Proj_1024"}

class AblationMLP(nn.Module):
    def __init__(self, cfg, emb_dim, ae=None):
        super().__init__()
        self.use_attention, self.use_disagreement, vm = cfg["attention"], cfg["disagreement"], cfg["verdict"]
        if self.use_attention: self.attn_pool = AttentionPool(emb_dim)
        if   vm == "raw":  self.verdict_repr = nn.Identity()
        elif vm == "dup":  self.verdict_repr = CyclicDuplication()
        elif vm == "ae":   self.verdict_repr = FrozenAEEncoder(ae)
        elif vm == "proj": self.verdict_repr = LearnableProjection()
        else: raise ValueError(vm)
        self.input_dim = emb_dim * (1 + self.use_attention + self.use_disagreement) + VERDICT_DIMS[vm]
        layers, prev = [nn.BatchNorm1d(self.input_dim)], self.input_dim
        for h in HIDDEN_DIMS:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(DROPOUT)]
            prev = h
        layers.append(nn.Linear(prev, OUTPUT_DIM))
        self.mlp = nn.Sequential(*layers)
    def forward(self, claim, traces, mask, scalars):
        with torch.autocast(device_type=claim.device.type, enabled=False):
            claim, traces, mask, scalars = claim.float(), traces.float(), mask.float(), scalars.float()
            parts = [claim]
            if self.use_attention:    parts.append(self.attn_pool(traces, mask))
            if self.use_disagreement: parts.append(masked_std(traces, mask))
            parts.append(self.verdict_repr(scalars))
            x = torch.cat(parts, dim=1)
        return self.mlp(x)

def pretrain_autoencoder(ARR):
    set_seed(42)
    ae = VerdictAutoencoder().to(DEVICE)
    Xtr, Xva = ARR["train"]["scalars"].to(DEVICE), ARR["val"]["scalars"].to(DEVICE)
    opt, mse = optim.Adam(ae.parameters(), lr=AE_LR), nn.MSELoss()
    loader = DataLoader(TensorDataset(ARR["train"]["scalars"]), batch_size=AE_BATCH_SIZE, shuffle=True,
                        generator=torch.Generator().manual_seed(42))
    best, best_state = float("inf"), None
    for ep in range(AE_EPOCHS):
        ae.train()
        for (xb,) in loader:
            xb = xb.to(DEVICE); opt.zero_grad(); loss = mse(ae(xb)[0], xb); loss.backward(); opt.step()
        ae.eval()
        with torch.no_grad():
            va = mse(ae(Xva)[0], Xva).item()
        if va < best: best, best_state = va, copy.deepcopy(ae.state_dict())
    ae.load_state_dict(best_state); ae.eval()
    print(f"  verdict autoencoder: best val reconstruction MSE {best:.2e}")
    return ae.cpu()

def _exp(idx, display_name, group, attention, disagreement, verdict):
    return {"idx": idx, "display": display_name, "group": group,
            "cfg": {"attention": attention, "disagreement": disagreement, "verdict": verdict}}
EXPERIMENTS = [
    _exp(1,  f"Full + Raw{SCALAR_DIM}",              "Full",            True,  True,  "raw"),
    _exp(2,  "Full + Duplication_1024",            "Full",            True,  True,  "dup"),
    _exp(3,  "Full + AE_1024",                     "Full",            True,  True,  "ae"),
    _exp(4,  "Full + Projection_1024",             "Full",            True,  True,  "proj"),
    _exp(5,  "No_Attention + Duplication_1024",    "No_Attention",    False, True,  "dup"),
    _exp(6,  "No_Attention + AE_1024",             "No_Attention",    False, True,  "ae"),
    _exp(7,  "No_Attention + Projection_1024",     "No_Attention",    False, True,  "proj"),
    _exp(8,  "No_Disagreement + Duplication_1024", "No_Disagreement", True,  False, "dup"),
    _exp(9,  "No_Disagreement + AE_1024",          "No_Disagreement", True,  False, "ae"),
    _exp(10, "No_Disagreement + Projection_1024",  "No_Disagreement", True,  False, "proj"),
]

@torch.no_grad()
def grid_eval(model, loader, w):
    model.eval(); loss_sum, w_sum, probs, labels = 0.0, 0.0, [], []
    for batch in loader:
        claim, traces, mask, scalars, y = [b.to(DEVICE, non_blocking=True) for b in batch]
        logits = model(claim, traces, mask, scalars).float()
        loss_sum += F.cross_entropy(logits, y, weight=w, reduction="sum").item(); w_sum += w[y].sum().item()
        probs.append(torch.softmax(logits, 1).cpu()); labels.append(y.cpu())
    return loss_sum / w_sum, torch.cat(probs).numpy(), torch.cat(labels).numpy()

def run_grid_experiment(ARR, exp, seed, ae, class_w, emb_dim):
    set_seed(seed)
    base = AblationMLP(exp["cfg"], emb_dim, ae=ae).to(DEVICE)
    model = nn.DataParallel(base) if N_GPUS > 1 else base
    opt = optim.Adam([p for p in base.parameters() if p.requires_grad], lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    w = class_w.to(DEVICE); crit = nn.CrossEntropyLoss(weight=w)
    sched = optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=LR_PLATEAU_FACTOR, patience=LR_PLATEAU_PATIENCE)
    amp = USE_AMP and DEVICE == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=amp)
    def loader(s, shuffle):
        a = ARR[s]; ds = TensorDataset(a["claim"], a["traces"], a["mask"], a["scalars"], a["labels"])
        if shuffle:
            return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True,
                              generator=torch.Generator().manual_seed(seed), pin_memory=DEVICE == "cuda")
        return DataLoader(ds, batch_size=256, shuffle=False, pin_memory=DEVICE == "cuda")
    tr, va, te = loader("train", True), loader("val", False), loader("test", False)
    best = {"f1": -1.0, "loss": float("inf"), "epoch": 0, "state": None}; patience = 0; hist = []; t0 = time.time()
    for ep in range(1, MAX_EPOCHS + 1):
        model.train(); tot = 0.0
        for batch in tr:
            claim, traces, mask, scalars, y = [b.to(DEVICE, non_blocking=True) for b in batch]
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=DEVICE, dtype=torch.float16, enabled=amp):
                loss = crit(model(claim, traces, mask, scalars).float(), y)
            if not torch.isfinite(loss):
                raise RuntimeError("non-finite training loss")
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(base.parameters(), GRAD_CLIP_NORM)
            scaler.step(opt); scaler.update(); tot += loss.item()
        vl, vp, vy = grid_eval(model, va, w); vf = mf1(vy, vp.argmax(1)); sched.step(vl)
        hist.append({"epoch": ep, "train_loss": tot / len(tr), "val_loss": vl, "val_macro_f1": vf})
        if vf > best["f1"] + 1e-6 or (abs(vf - best["f1"]) <= 1e-6 and vl < best["loss"]):
            best = {"f1": vf, "loss": vl, "epoch": ep, "state": copy.deepcopy(base.state_dict())}; patience = 0
        else:
            patience += 1
            if patience >= EARLY_STOP_PATIENCE:
                break
    base.load_state_dict(best["state"])
    _, vp, vy = grid_eval(model, va, w); _, tp, ty = grid_eval(model, te, w)
    f = f1s(ty, tp.argmax(1))
    res = {"best_epoch": best["epoch"], "epochs_run": len(hist), "seconds": time.time() - t0, "history": hist,
           "val_macro_f1": mf1(vy, vp.argmax(1)), "val_acc": float(np.mean(vp.argmax(1) == vy)),
           "test_macro_f1": float(f.mean()), "test_acc": float(np.mean(tp.argmax(1) == ty)),
           "test_f1": f.tolist(), "input_dim": base.input_dim}
    del model, base, opt; gc.collect()
    if DEVICE == "cuda": torch.cuda.empty_cache()
    return res, vp, tp

GRID_DIR = os.path.join(OUTPUT_DIR, "grid" + ("_trial" if LIMIT_ROWS else ""))
GRID_ROWS = []
if RUN_GRID:
    os.makedirs(GRID_DIR, exist_ok=True)
    if INPUT_ROOT and os.path.isdir(INPUT_ROOT):          # reuse runs of an attached earlier version
        for p in glob.glob(os.path.join(INPUT_ROOT, "**", os.path.basename(GRID_DIR), "*.json"), recursive=True):
            dst = os.path.join(GRID_DIR, os.path.basename(p))
            if not os.path.exists(dst): shutil.copy(p, dst)
    _cnt = np.bincount(Y["train"], minlength=3)
    class_w = torch.tensor(len(Y["train"]) / (3.0 * np.maximum(_cnt, 1)), dtype=torch.float32)
    for set_label, tag in TRACE_SETS.items():
        needed = [(e, sd) for sd in GRID_SEEDS for e in EXPERIMENTS
                  if not os.path.exists(f"{GRID_DIR}/{tag}__{e['idx']:02d}__seed{sd}.json")]
        ARR = None
        if needed:
            print(f"\n===== trace embeddings: {set_label} ({tag}) — {len(needed)} run(s) =====", flush=True)
            ARR = build_arrays(tag)
            if ARR is None:
                print(f"  {tag}: files not found, this embedding set is skipped"); continue
            ae = pretrain_autoencoder(ARR); emb_dim = ARR["train"]["claim"].shape[1]
            for e, sd in needed:
                res, vp, tp = run_grid_experiment(ARR, e, sd, ae, class_w, emb_dim)
                res.update({"set": set_label, "tag": tag, "idx": e["idx"], "display": e["display"], "group": e["group"],
                            "verdict": VERDICT_NAMES[e["cfg"]["verdict"]], "seed": sd})
                np.savez(f"{GRID_DIR}/{tag}__{e['idx']:02d}__seed{sd}.npz", val=vp, test=tp)
                with open(f"{GRID_DIR}/{tag}__{e['idx']:02d}__seed{sd}.json", "w") as f: json.dump(res, f)
                print(f"  seed {sd:3d} | {e['idx']:2d}. {e['display']:36s} | best ep {res['best_epoch']:2d}/{res['epochs_run']:2d} | "
                      f"val {res['val_macro_f1']:.4f} | test {res['test_macro_f1']:.4f} | acc {res['test_acc']:.4f} | "
                      f"{res['seconds']:.0f}s", flush=True)
            del ARR, ae; gc.collect()
        for p in sorted(glob.glob(f"{GRID_DIR}/{tag}__*.json")):
            GRID_ROWS.append(json.load(open(p)))


===== trace embeddings: no instruction (qwen3emb8b_en_v1) — 30 run(s) =====
  qwen3emb8b_en_v1 / train: 6400 rows, dim 4096, traces used mean 19.85
  qwen3emb8b_en_v1 / val: 1600 rows, dim 4096, traces used mean 15.00
  qwen3emb8b_en_v1 / test: 2558 rows, dim 4096, traces used mean 20.00
  verdict autoencoder: best val reconstruction MSE 5.78e-07
  seed  42 |  1. Full + Raw6                          | best ep 12/18 | val 0.5958 | test 0.4867 | acc 0.6974 | 143s
  seed  42 |  2. Full + Duplication_1024              | best ep  2/ 8 | val 0.5847 | test 0.5789 | acc 0.7748 | 66s
  seed  42 |  3. Full + AE_1024                       | best ep 12/18 | val 0.5908 | test 0.4976 | acc 0.7283 | 151s
  seed  42 |  4. Full + Projection_1024               | best ep  2/ 8 | val 0.5946 | test 0.5688 | acc 0.7447 | 73s
  seed  42 |  5. No_Attention + Duplication_1024      | best ep  4/10 | val 0.5822 | test 0.5400 | acc 0.7005 | 65s
  seed  42 |  6. No_Attention + AE_1024               | best ep  4/1

## 6. Grid results
Mean ± std over seeds. The majority-vote row is the reference: a configuration is only useful if it is above it by
clearly more than its own std.

In [10]:
if GRID_ROWS:
    G = pd.DataFrame([{k: v for k, v in r.items() if k != "history"} for r in GRID_ROWS])
    G["test_f1_conf"] = G["test_f1"].apply(lambda x: x[2])
    G.drop(columns=["test_f1"]).to_csv(os.path.join(OUTPUT_DIR, "grid_runs.csv"), index=False)
    agg = (G.groupby(["set", "idx", "display", "group", "verdict"], sort=False)
             .agg(seeds=("seed", "count"), val_f1=("val_macro_f1", "mean"), test_f1=("test_macro_f1", "mean"),
                  test_f1_std=("test_macro_f1", "std"), test_acc=("test_acc", "mean"),
                  test_f1_conf=("test_f1_conf", "mean"), best_ep=("best_epoch", "mean")).reset_index()
             .sort_values(["set", "idx"]))
    maj = mf1(Y["test"], majority(D["test"]["counts"]))
    agg["Δ vs majority"] = agg["test_f1"] - maj
    agg.to_csv(os.path.join(OUTPUT_DIR, "grid_summary.csv"), index=False)
    print(f"Majority vote of the traces (reference): test Macro-F1 {maj:.4f}, "
          f"acc {np.mean(majority(D['test']['counts']) == Y['test']):.4f}")
    for set_label in agg["set"].unique():
        a = agg[agg["set"] == set_label].drop(columns=["set", "group", "verdict"]).rename(columns={
            "idx": "#", "display": "Configuration", "val_f1": "Val Macro-F1", "test_f1": "Test Macro-F1",
            "test_f1_std": "± std", "test_acc": "Test Acc", "test_f1_conf": "F1 Conflicting", "best_ep": "Best ep"})
        print(f"\nTrace embeddings: {set_label}")
        try:
            display(a.style.format({c: "{:.4f}" for c in ["Val Macro-F1", "Test Macro-F1", "± std", "Test Acc", "F1 Conflicting"]}
                                   | {"Δ vs majority": "{:+.4f}", "Best ep": "{:.1f}"}).hide(axis="index")
                    .highlight_max(subset=["Val Macro-F1", "Test Macro-F1"], color="#c8e6c9"))
        except Exception:
            print(a.round(4).to_string(index=False))
        m = (agg[agg["set"] == set_label].pivot(index="group", columns="verdict", values="test_f1")
             .reindex(index=["Full", "No_Attention", "No_Disagreement"], columns=list(VERDICT_NAMES.values())))
        print("Test Macro-F1 matrix (rows = feature configuration, columns = verdict representation):")
        print(m.round(4).to_string(na_rep="—"))
        bv = agg[agg["set"] == set_label].sort_values("val_f1").iloc[-1]
        print(f"Chosen by validation: {bv['display']} → test Macro-F1 {bv['test_f1']:.4f} ± {bv['test_f1_std']:.4f} "
              f"({bv['Δ vs majority']:+.4f} vs majority vote)")
    if agg["set"].nunique() == 2:
        s0, s1 = list(TRACE_SETS)[0], list(TRACE_SETS)[1]
        p = agg.pivot(index="display", columns="set", values="test_f1").reindex([e["display"] for e in EXPERIMENTS])
        sd = agg.pivot(index="display", columns="set", values="test_f1_std").reindex(p.index)
        cmp = pd.DataFrame({s0: p[s0], s1: p[s1], "Δ (instruction − none)": p[s1] - p[s0],
                            "larger std of the two": sd.max(axis=1)})
        print(f"\nInstruction vs no instruction, test Macro-F1 per configuration (mean over seeds):")
        print(cmp.round(4).to_string())
        print(f"Mean Δ over the 10 configurations: {cmp.iloc[:, 2].mean():+.4f}")
        cmp.to_csv(os.path.join(OUTPUT_DIR, "grid_instruction_vs_none.csv"))
else:
    print("no grid runs available")
print(f"\ntotal time {(time.time() - T_START) / 60:.1f} min")

Majority vote of the traces (reference): test Macro-F1 0.5967, acc 0.7721

Trace embeddings: no instruction


#,Configuration,seeds,Val Macro-F1,Test Macro-F1,± std,Test Acc,F1 Conflicting,Best ep,Δ vs majority
1,Full + Raw6,3,0.5949,0.4937,0.0323,0.6906,0.1965,10.0,-0.1030
2,Full + Duplication_1024,3,0.5832,0.5745,0.0235,0.7469,0.2381,3.0,-0.0222
3,Full + AE_1024,3,0.5902,0.5376,0.0355,0.7262,0.2170,8.0,-0.0591
4,Full + Projection_1024,3,0.5903,0.5627,0.0057,0.7355,0.2249,3.7,-0.0340
5,No_Attention + Duplication_1024,3,0.5802,0.5180,0.0320,0.7041,0.1831,6.3,-0.0786
6,No_Attention + AE_1024,3,0.5876,0.5371,0.0097,0.6917,0.2042,3.7,-0.0595
7,No_Attention + Projection_1024,3,0.5837,0.4931,0.0085,0.6827,0.1802,8.3,-0.1035
8,No_Disagreement + Duplication_1024,3,0.5928,0.5124,0.0340,0.7248,0.2140,11.7,-0.0843
9,No_Disagreement + AE_1024,3,0.5902,0.5263,0.0395,0.7095,0.2010,5.3,-0.0703
10,No_Disagreement + Projection_1024,3,0.5920,0.5404,0.0304,0.7230,0.2240,7.7,-0.0563


Test Macro-F1 matrix (rows = feature configuration, columns = verdict representation):
verdict           Raw 6  Dup_1024  AE_1024  Proj_1024
group                                                
Full             0.4937    0.5745   0.5376     0.5627
No_Attention          —    0.5180   0.5371     0.4931
No_Disagreement       —    0.5124   0.5263     0.5404
Chosen by validation: Full + Raw6 → test Macro-F1 0.4937 ± 0.0323 (-0.1030 vs majority vote)

total time 363.5 min
